# 03 - Dependency graph and provisional risk score

**Purpose:** Show every dependency edge with provenance, conflicts and cycles, the additive risk score and its sensitivity.

**Inputs:** cryptoshift.planning (demo configs + synthetic descriptor), planning_config.json

**Outputs:** graph figure, edge table, risk contribution chart, sensitivity heatmap

**Status labels:** Graph VERIFIED on SYNTHETIC inputs. Risk score PROVISIONAL and UNVALIDATED (weights are assumptions).

Context: the demo bank in `demo_bank_app/` is SYNTHETIC. Charts are also saved to `scratch/derived/member3/figs/`.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys, json, subprocess
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "cryptoshift").is_dir())
for p in (str(ROOT / "src"), str(ROOT)):
    if p not in sys.path:
        sys.path.insert(0, p)
DERIVED = ROOT / "scratch" / "derived" / "member3"
FIGS = DERIVED / "figs"
FIGS.mkdir(parents=True, exist_ok=True)
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
PY = sys.executable

def ensure(rel_path, script, *args):
    """Run a project script once if its output file is missing (keeps notebooks runnable in a fresh checkout)."""
    target = ROOT / rel_path
    if not target.is_file():
        print(f"{rel_path} missing -> running {script} ...")
        subprocess.run([PY, str(ROOT / "scripts" / script), *args], cwd=ROOT, check=True)
    return target

def savefig(name):
    plt.tight_layout(); plt.savefig(FIGS / f"{name}.png", dpi=120); plt.show()
# This notebook reads no credentials. If one is ever needed, read it from an environment variable at runtime and
# never display it.

In [ ]:
import networkx as nx
from matplotlib.lines import Line2D
from cryptoshift.planning.planner import build_problem, sensitivity
p = build_problem()
colors = {'OBSERVED':'#2a9d8f','DECLARED':'#8d99ae','INFERRED':'#f4a261'}
g = nx.MultiDiGraph(); g.add_nodes_from(p['services'])
for e in p['edges']: g.add_edge(e['upstream'], e['downstream'], prov=e['provenance'])
pos = {'auth_gateway':(0,1),'ledger_audit':(0,-1),'interbank_tls':(1,-1.4),'card_vault':(1.2,1.4),'payment_signing':(1.2,0.2),'settlement_batch':(2.4,-0.4)}
pos = {n: pos.get(n, (3, 0)) for n in g.nodes}
fig, ax = plt.subplots(figsize=(10, 5))
node_col = ['#2a9d8f' if p['services'][n]['evidence_source']=='OBSERVED' else '#e9c46a' for n in g.nodes]
nx.draw_networkx_nodes(g, pos, node_color=node_col, node_size=4200, ax=ax)
nx.draw_networkx_labels(g, pos, labels={n: n.replace('_', '\n') for n in g.nodes}, font_size=8, ax=ax)
for i, (u, v, d) in enumerate(g.edges(data=True)):
    rad = 0.15 if d['prov']=='OBSERVED' else -0.15
    nx.draw_networkx_edges(g, pos, edgelist=[(u, v)], edge_color=colors[d['prov']], width=2.2, arrowsize=18, connectionstyle=f'arc3,rad={rad}', node_size=4200, ax=ax)
ax.margins(0.12)
ax.legend(handles=[Line2D([0],[0], color=c, lw=3, label=k) for k, c in colors.items()] + [Line2D([0],[0], marker='o', color='w', markerfacecolor='#e9c46a', markersize=12, label='DECLARED_ONLY service')], loc='lower right')
ax.set_title('Service dependencies (upstream -> downstream) by provenance'); ax.axis('off')
savefig('03_dependency_graph')

In [ ]:
ga = p['graph_analysis']; print('cycles:', ga['cycles'] or 'none', '| edge counts:', ga['edge_counts'])
display(pd.DataFrame(ga['conflicts']))
pd.DataFrame(p['edges'])[['upstream','downstream','provenance','file','line','reason']]

In [ ]:
contrib = pd.DataFrame({s: v['risk']['contributions'] for s, v in p['services'].items()}).T
contrib = contrib.loc[contrib.sum(axis=1).sort_values().index]
ax = contrib.plot(kind='barh', stacked=True, figsize=(10, 4), colormap='viridis', title='Provisional risk score = sum of weighted contributions (UNVALIDATED)')
ax.set_xlabel('score'); savefig('03_risk_contributions')
contrib.assign(total=contrib.sum(axis=1)).round(2)

In [ ]:
sa = sensitivity(p)
runs = pd.DataFrame(sa['runs'])
heat = runs.pivot(index='weight', columns='factor', values='kendall_tau')
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
im = ax[0].imshow(heat.values, cmap='RdYlGn', vmin=-1, vmax=1)
ax[0].set_xticks(range(len(heat.columns)), [f'x{c}' for c in heat.columns]); ax[0].set_yticks(range(len(heat.index)), heat.index)
for (i, j), v in __import__('numpy').ndenumerate(heat.values): ax[0].text(j, i, f'{v:.2f}', ha='center', va='center')
ax[0].set_title('Rank stability (Kendall tau vs base)'); ax[0].grid(False); fig.colorbar(im, ax=ax[0])
pd.Series({'rank changed': sa['rank_changed_runs'], 'rank unchanged': sa['n_runs']-sa['rank_changed_runs'], 'schedule changed': sa['schedule_changed_runs'], 'schedule unchanged': sa['n_runs']-sa['schedule_changed_runs']}).plot(kind='bar', ax=ax[1], color=['#e76f51','#2a9d8f','#e76f51','#2a9d8f'], title=f"{sa['n_runs']} weight perturbations (x0.5 / x1.5)")
ax[1].tick_params(axis='x', rotation=20); savefig('03_sensitivity')